# Raw Data Extraction

- Setting up GCP buckets and BigQuery
- Uploading a sample of articles to a GCP Bucket
- Uploading Prices from `yfinance`

In [ ]:
import os
import sys
from pathlib import Path

sys.path.append(
    Path.cwd().parents[1].as_posix()
)

In [ ]:
import google.auth
import pandas as pd
import polars as pl
import yfinance as yf

from google.cloud import bigquery
from google.cloud import bigquery_storage_v1
from google.cloud import storage
from tqdm.auto import tqdm

from finnews.data.controllers import FNSPIDController

# Defines

In [ ]:
dc = FNSPIDController()

In [ ]:
root_input_dir = os.path.join("data", "raw")

In [ ]:
_, PROJECT_ID = google.auth.default()
REGION = "europe-west4"
BUCKET_NAME = "fin-news"
RAW_DATA_DIR = "raw/articles"
DATASET_ID = BUCKET_NAME.replace("-", "_")

In [ ]:
# BigQuery client
bigquery_client = bigquery.Client(
    project=PROJECT_ID,
    location=REGION
)

bqstorage = bigquery_storage_v1.BigQueryReadClient()

In [ ]:
symbols_sample = [
    'MMM',
    'AMZN',
    'AXP',
    'AMGN',
    'AAPL',
    'BA',
    'CAT',
    'CVX',
    'CSCO',
    'KO',
    'GS',
    'HD',
    'HON',
    'IBM',
    'JNJ',
    'JPM',
    'MCD',
    'MRK',
    'MSFT',
    'NKE',
    'NVDA',
    'PG',
    'CRM',
    'SHW',
    'TRV',
    'UNH',
    'VZ',
    'V',
    'WMT',
    'DIS',

    # A few others
    'GOOG',
    'TSLA',
    'FB',
    'META'
]

In [ ]:
start_date = '2010-01-01'
end_date = '2024-06-01'

# GCP Bucket Setup

In [ ]:
client = storage.Client(project=PROJECT_ID)

In [ ]:
bucket = client.get_bucket(BUCKET_NAME)

if bucket is None:
    bucket = client.create_bucket(
        bucket_or_name=BUCKET_NAME,
        location=REGION
    )

# Raw FNSPID Articles

Downloading raw FNSPID data using data controller.

In [ ]:
dc.download_raw_data(
    output_dir=root_input_dir
)

# Articles

Selecting a sample of articles about stocks in `symbols_sample`.

In [ ]:
dl = dc.get_articles(root_input_dir)

In [ ]:
dl.collect_schema().names()

In [ ]:
# Removing null Articles
dl = dl.filter(
    (~pl.col("Article").is_null())
    & (pl.col("date") >= pd.to_datetime(start_date).to_pydatetime())
)

dl = dl.select(
    ["date", "Stock_symbol", "Article"]
).rename(
    {
        "Stock_symbol": "symbol",
        "Article": "article"
    }
)

In [ ]:
symbols_iter = tqdm(symbols_sample[28:])

# Deliberately done in a less efficient manner w/o partitioning
for cur_symbol in symbols_iter:
    symbols_iter.set_description_str(cur_symbol)
    try:
        dl_cur = dl.filter(pl.col("symbol") == cur_symbol)
    
        dl_cur.sink_parquet(
            f"gs://{BUCKET_NAME}/{RAW_DATA_DIR}/{cur_symbol}.parquet",
            storage_options={"token": "google_default"},
        )
    except Exception as e:
        print(f'Exception with {cur_symbol}: {e}')

# Prices

Saving prices from yfinance directly to BigQuery.

In [ ]:
df_prices_raw = yf.download(
    tickers=symbols_sample,
    start=start_date,
    end=end_date,
    group_by='ticker',
    keepna=True
)

In [ ]:
df_prices = df_prices_raw.stack(0, future_stack=True)
df_prices = df_prices.reset_index()
df_prices.columns = [x.lower() for x in df_prices.columns]
df_prices = df_prices.drop(columns=['adj close'])

# Correcting META / FB
df_prices = df_prices.loc[
    df_prices['ticker'] != 'FB'
]
df_prices['ticker'] = df_prices['ticker'].replace({'META': 'FB'})

# Sorting
df_prices = df_prices.sort_values(by=['ticker', 'date'], ignore_index=True)

# Replacing leading NaNs
df_prices['mask'] = df_prices.groupby('ticker')['close'].transform(
    lambda x: x.notna().cummax()
)

df_prices = df_prices[df_prices['mask']].reset_index(drop=True)
df_prices = df_prices.drop(columns=['mask'])

In [ ]:
# Writing to BigQuery
FULL_PRICE_TABLE_ID = f"{PROJECT_ID}.{DATASET_ID}.prices"

bigquery_client.load_table_from_dataframe(
    dataframe=df_prices,
    destination=FULL_PRICE_TABLE_ID
)